# CS-781 Phase 1 — Architecture 4: Efficient Modern Causal Decoder (~99.31M, 400M tokens) — **v3 BEST**

Same architecture as before (17 blocks, d=640, 10 query / 2 KV heads GQA, RoPE, RMSNorm, SwiGLU, tied embeddings, ~99.31M parameters), same
assignment BPE tokenizer and same 400M-token corpus. **Cells 3, 4, 5, 7 (tokenizer + data) are byte-for-byte unchanged.**
v1 weights / checkpoints load into v2 unchanged (same `state_dict` keys, same maths — verified numerically).

## What changed in v2

| # | Change | Why |
|---|--------|-----|
| 1 | **Bug fix (Cell 9):** `RotaryEmbedding` had a method called `apply`, which overrides `nn.Module.apply` | `self.apply(self._init_weights)` then raised `TypeError`: **v1 could not build the model at all** |
| 2 | RoPE cos/sin gathered once per forward; no per-layer GPU→CPU sync | v1 did 34 gathers + 17 syncs per step |
| 3 | GQA-native cached decoding (no 5× K/V expansion per token); chunks after a cache are causal | Faster generation, which makes multi-candidate decoding affordable |
| 4 | Cell 9b self-test: padded batched generation == un-padded single run, GQA path == expanded path, chunked prefill | The exact failure modes that silently hurt generation quality |
| 5 | `build_model()`; Cell 11 starts from fresh weights; no weight decay on the tied embedding | Removes the fragile "rerun Cell 9 after the pilot" step |
| 6 | **Decoding (Cells 12/14/15):** no-repeat n-gram blocking, per-row temperature, multiple candidates from one prefill, **MBR selection** with BERTScore utility | Decoding is the largest remaining lever on BERTScore; every setting is now scored by BERTScore (v1 screened by ROUGE-L) with a paired noise estimate |
| 7 | 128th word is only accepted once complete | v1 could cut the last word in the middle |

## Workflow
1. Cells 1–9b (Cell 9b must print `SELF-TEST PASSED`).
2. Cell 10 `pilot` → Cell 11 for ~100–150 steps; check loss, tokens/s, VRAM.
3. Set `RUN_MODE = "full"` in Cell 10 → rerun Cell 10 and Cell 11.
4. Cell 12 (local generation) → Cell 13 (metrics) → **Cell 14 (decoding sweep)** → Cell 15 with `RUN_OFFICIAL_TEST = True`
   (Cell 15 reads `best_decode.json` written by Cell 14).


### v3 review decisions

I reviewed the v2 changes against the Phase-1 objective and kept the changes that are likely to help reliability, speed, or final BERTScore **without changing the legal next-token objective**:

- shared RoPE table gathered once per forward;
- native compact-GQA cached decoding;
- correct causality for cached multi-token chunks;
- much stricter generation-path self-tests;
- fresh-model construction inside the training cell;
- no weight decay on the tied token embedding;
- on-GPU loss accumulation;
- no-repeat n-gram support (off by default and tested only in decoding sweep);
- complete-word stopping for the 128th word;
- BERTScore on **every validation decoding setting**, not just a ROUGE shortlist;
- optional multi-candidate consensus decoding.

One v2 idea is intentionally changed: **BERTScore/RoBERTa is not used inside official generation to choose among candidates.**
BERTScore is used only as the held-out evaluation metric for choosing decoding hyperparameters. Optional multi-candidate
selection uses **ROUGE-L consensus only**, so official inference remains based solely on this scratch-trained model and
non-pretrained text-overlap logic.


In [1]:
# Cell 1 — Install dependencies
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ijson",
    "huggingface_hub",
    "transformers==4.48.3",
    "bert-score==0.3.13",
])

print("Dependencies ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.1 MB/s eta 0:00:00
Dependencies ready.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Cell 2 — Imports + EfficientDecoder-100M configuration   (v2: decoding options added; tokenizer/data settings untouched)
import os
if os.path.isdir("/kaggle/temp"):
    os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")

import re, gc, json, math, time, random, zipfile, heapq, shutil
from pathlib import Path
from collections import Counter

import ijson
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, HfApi, login

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded.")
except Exception as e:
    print("HF secret unavailable; public access will be attempted.", repr(e))

if HF_TOKEN:
    login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILES = {
    "sc":   "Phase_1_Pretraining/Train/train_supreme_court.json",
    "hc":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "tr":   "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}

# ----------------------------- output paths
ROOT = Path("/kaggle/working/cs781_effdecoder_100m")
DATA_DIR = ROOT / "data"
ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_PATH = DATA_DIR / "tokenizer.json"
TRAIN_BIN = DATA_DIR / "train_all.bin"
VAL_BIN = DATA_DIR / "val_all.bin"
META_PATH = DATA_DIR / "meta.json"
VAL_GEN_PATH = DATA_DIR / "val_gen.json"
VAL_REF_PATH = DATA_DIR / "val_ref.json"

# OPTIONAL: reuse a compatible tokenizer + tokenized binaries.
# This must come from the SAME 24,576-token BPE design and same Phase-1 corpus.
REUSE_DATA_FROM = None

if REUSE_DATA_FROM:
    reuse_dir = Path(REUSE_DATA_FROM)
    for _name in [
        "tokenizer.json", "census.json", "meta.json",
        "train_all.bin", "val_all.bin", "val_gen.json", "val_ref.json"
    ]:
        _src, _dst = reuse_dir / _name, DATA_DIR / _name
        if _src.exists() and not _dst.exists():
            shutil.copy2(_src, _dst)
            print("Reused:", _name)

# Set in run-mode cell.
RUN_NAME = None
CKPT_DIR = None
LATEST_CKPT = None
BEST_CKPT = None
FINAL_CKPT = None
OUTSIDE_LATEST_CKPT = None
OUTSIDE_BEST_CKPT = None

HF_CHECKPOINT_REPO = None
HF_UPLOAD_EVERY = 2000
HF_SQUASH_EVERY_N_UPLOADS = 4

LOCAL_PRED = ROOT / "val_pred.json"
OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# ----------------------------- tokenizer
VOCAB_SIZE = 24_576
TOKENIZER_CHAR_BUDGET = {
    "sc":   16_000_000,
    "hc":   16_000_000,
    "tr":   10_000_000,
    "acts":  6_000_000,
}
TOKENIZER_MIN_COUNT = 2

# ----------------------------- 400M training mix
TRAIN_TOKEN_BUDGET = {
    "sc":   144_000_000,
    "hc":   144_000_000,
    "tr":    72_000_000,
    "acts":  40_000_000,
}
TARGET_TOTAL_TOKENS = sum(TRAIN_TOKEN_BUDGET.values())

VAL_TOKENS_PER_SOURCE = 500_000
VAL_GEN_PER_SOURCE = 40
VAL_PERMILLE = 5
DELETE_SOURCE_AFTER_STAGE = True
ENCODE_PROCESSES = min(4, os.cpu_count() or 2)

# ----------------------------- model
SEQ_LEN = None
MAX_POSITIONS = 2560

D_MODEL = 640
N_LAYERS = 17
N_HEADS = 10
N_KV_HEADS = 2
HEAD_DIM = D_MODEL // N_HEADS
FFN_DIM = 2048
ROPE_BASE = 10_000.0
RMS_EPS = 1e-5
DROPOUT = 0.0

PROMPT_P10_TOKENS = None
PROMPT_P95_TOKENS = None
EXPECTED_CONT_TOKENS = None
PROMPT_SPLIT_MIN = None
PROMPT_SPLIT_MAX = None
PROMPT_KEEP_TOKENS = None

# ----------------------------- optimization
# 100M + 17 layers is heavier than Model 2. 2x8 keeps the same effective
# 16-block optimizer batch while being safe on a 16GB Tesla T4.
MICRO_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 3e-4
MIN_LR_RATIO = 0.10
WARMUP_RATIO = 0.02
WEIGHT_DECAY = 0.10
GRAD_CLIP = 1.0

LR_SCHEDULE = "wsd"
WSD_DECAY_FRAC = 0.20

LOSS_ON_ALL_POSITIONS = True
TRAIN_BLOCK_LIMIT = None

LOG_EVERY = 50
VAL_EVERY = 1000
SAVE_EVERY = 500
VAL_EVAL_BLOCKS = 256

# ----------------------------- generation
TARGET_GENERATION_WORDS = 128
GEN_TEMPERATURE = 0.8
GEN_TOP_P = 0.95
GEN_TOP_K = 0
GEN_REP_PENALTY = 1.03
GEN_NO_REPEAT_NGRAM = 0          # v2: block repeated token n-grams inside the generated text (0 = off, else >= 2)
GEN_MAX_NEW_TOKENS = 1280
GEN_BATCH_SIZE = 16              # prompts per generation batch (single-sample decoding)

# v2: Minimum-Bayes-Risk decoding. Sample MBR_N candidates per prompt (one prefill), keep the candidate that is
# most similar (BERTScore) to the others. MBR_N = 1 switches it off. Cell 14 decides whether it helps.
MBR_N = 1
MBR_INCLUDE_GREEDY = True        # candidate 0 of each prompt is decoded greedily
MBR_UTILITY = "rougel"           # official-safe consensus: NO pretrained evaluator inside generation
MAX_DECODE_ROWS = 128            # cap on (prompts x candidates) decoded together
USE_SWEEP_BEST = True            # Cell 15 uses ROOT/best_decode.json written by Cell 14 when it exists
OFFICIAL_PARTIAL_EVERY = 4
GENERATION_SEED = SEED + 4040

FAST_EVAL_NO_BERT = False
BERTSCORE_BATCH_SIZE = 8
RUN_OFFICIAL_TEST = False

print("Output:", ROOT)
print("Target train tokens:", f"{TARGET_TOTAL_TOKENS/1e6:.0f}M")
print("Architecture target: ~99M modern GQA/RoPE/RMSNorm/SwiGLU decoder")


Device: cuda
GPU: Tesla T4
HF token loaded.
Output: /kaggle/working/cs781_effdecoder_100m
Target train tokens: 400M
Architecture target: ~99M modern GQA/RoPE/RMSNorm/SwiGLU decoder


In [3]:
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

Tokenizer class ready.


In [4]:
# Cell 4 — Streaming source utilities + robust test discovery
def iter_docs(path):
    """Yield document text from the official top-level JSON array."""
    with open(path, "rb") as f:
        for obj in ijson.items(f, "item"):
            text = obj.get("text", "")
            if isinstance(text, str) and text.strip():
                yield text

def fetch_source(key):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=SOURCE_FILES[key],
        repo_type="dataset",
        token=HF_TOKEN,
    )

def find_test_file():
    files = HfApi().list_repo_files(
        HF_DATASET_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    candidates = [
        f for f in files
        if f.startswith("Phase_1_Pretraining/Test") and f.endswith(".json")
    ]
    if not candidates:
        raise RuntimeError("Could not discover Phase-1 test JSON.")
    chosen = candidates[0]
    print("Official test file:", chosen)
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=chosen,
        repo_type="dataset",
        token=HF_TOKEN,
    )

def maybe_delete_download(path):
    if not DELETE_SOURCE_AFTER_STAGE:
        return
    try:
        real = os.path.realpath(path)
        if os.path.exists(real):
            os.remove(real)
            print("Deleted cached source to save disk:", real)
    except Exception as e:
        print("Could not delete source cache:", repr(e))

def source_census(key, sample_docs=1000):
    path = fetch_source(key)
    size = os.path.getsize(path)
    n = chars = words = 0
    lens = []
    for doc in iter_docs(path):
        n += 1
        chars += len(doc)
        w = len(doc.split())
        words += w
        lens.append(w)
        if n >= sample_docs:
            break
    avg_chars = chars / max(n,1)
    # JSON overhead correction; only used to choose approximate uniform sampling probability.
    est_docs = int(size / max(1.0, avg_chars * 1.03))
    est_chars = avg_chars * est_docs
    return {
        "file_bytes":size,
        "sample_docs":n,
        "avg_chars":avg_chars,
        "avg_words":words/max(n,1),
        "est_docs":est_docs,
        "est_chars":est_chars,
        "p10_p50_p90_words":[int(np.percentile(lens,q)) for q in (10,50,90)] if lens else [],
    }

print("Streaming utilities ready.")

Streaming utilities ready.


In [5]:
# Cell 5 — Census + train BPE on a uniform sample from ALL FOUR sources
CENSUS_PATH = DATA_DIR / "census.json"

if CENSUS_PATH.exists():
    census = json.load(open(CENSUS_PATH))
    print("Loaded existing census.")
else:
    census = {}
    for key in SOURCE_FILES:
        t0=time.time()
        census[key]=source_census(key, sample_docs=1000)
        print(
            key,
            f"{census[key]['file_bytes']/1e9:.2f} GB",
            "avg_chars", f"{census[key]['avg_chars']:,.0f}",
            "est_docs", f"{census[key]['est_docs']:,}",
            "words p10/50/90", census[key]["p10_p50_p90_words"],
            f"({time.time()-t0:.0f}s)"
        )
        # Keep downloaded file for tokenizer stage in this same session.
    json.dump(census, open(CENSUS_PATH,"w"), indent=2)

if TOKENIZER_PATH.exists():
    tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
    print("Loaded existing tokenizer:", len(tokenizer))
else:
    rng = random.Random(SEED)

    def tokenizer_text_stream():
        for key, char_budget in TOKENIZER_CHAR_BUDGET.items():
            path = fetch_source(key)
            est_chars = max(1.0, census[key]["est_chars"])
            # Slight oversampling makes it unlikely we finish far below target.
            q = min(1.0, 1.05 * char_budget / est_chars)
            got = 0
            seen = 0
            print(f"[{key}] tokenizer sample probability={q:.6f}")
            for doc in iter_docs(path):
                seen += 1
                if rng.random() < q:
                    # Cap unusually huge documents so one case cannot dominate merge learning.
                    d = doc[:30000]
                    got += len(d)
                    yield d
                    if got >= char_budget:
                        break
            print(f"[{key}] tokenizer chars={got/1e6:.1f}M | docs scanned={seen:,}", flush=True)
            # Do NOT delete here: Cell 7 re-reads this file (all four are already on disk after the census),
            # so keeping it avoids downloading every source twice. Cell 7 deletes each file when finished.

    tokenizer = LegalBPE(VOCAB_SIZE)
    t0=time.time()
    tokenizer.fit(tokenizer_text_stream(), min_count=TOKENIZER_MIN_COUNT)
    tokenizer.save_json(TOKENIZER_PATH)
    print(f"Tokenizer saved to {TOKENIZER_PATH}")
    print(f"Tokenizer training elapsed: {(time.time()-t0)/60:.1f} min")

# Exact byte-level round-trip sanity.
probe = "IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
roundtrip = tokenizer.decode(tokenizer.encode(probe))
print("Round-trip exact:", roundtrip == probe)
print("Vocab:", len(tokenizer))

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

sc 1.32 GB avg_chars 36,184 est_docs 35,486 words p10/50/90 [1580, 3869, 11757] (9s)


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

hc 10.67 GB avg_chars 2,648 est_docs 3,913,049 words p10/50/90 [68, 206, 960] (37s)


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

tr 6.48 GB avg_chars 27,783 est_docs 226,580 words p10/50/90 [238, 3432, 9831] (36s)


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

acts 0.73 GB avg_chars 44,418 est_docs 15,929 words p10/50/90 [407, 2827, 14848] (9s)
[sc] tokenizer sample probability=0.013084
  BPE corpus read: 10M chars | unique units=78,714
[sc] tokenizer chars=11.1M | docs scanned=55,528
[hc] tokenizer sample probability=0.001621
  BPE corpus read: 20M chars | unique units=146,851
[hc] tokenizer chars=13.8M | docs scanned=2,536,738
[tr] tokenizer sample probability=0.001668
  BPE corpus read: 30M chars | unique units=216,937
[tr] tokenizer chars=9.0M | docs scanned=869,132
[acts] tokenizer sample probability=0.008904
[acts] tokenizer chars=2.9M | docs scanned=18,822
BPE training units after min_count=2: 103,942
Counting time: 1.5 min
merge 1/24,316 | freq=760,292 | vocab=261
merge 1,000/24,316 | freq=2,684 | vocab=1,260
merge 2,000/24,316 | freq=1,112 | vocab=2,260
merge 3,000/24,316 | freq=635 | vocab=3,260
merge 4,000/24,316 | freq=415 | vocab=4,260
merge 5,000/24,316 | freq=297 | vocab=5,260
merge 6,000/24,316 | freq=225 | vocab=6,260
merge 

In [6]:
# Cell 6 — Measure OFFICIAL 512-word prompts and choose the real decoder context
test_path = find_test_file()
test_data_for_stats = json.load(open(test_path, encoding="utf-8"))

prompt_token_lens = []
prompt_word_lens = []

for rec in test_data_for_stats.values():
    text = rec["text"]
    prompt_token_lens.append(len(tokenizer.encode(text)))
    prompt_word_lens.append(len(text.split()))

tpw = sum(prompt_token_lens) / max(1, sum(prompt_word_lens))

PROMPT_P10_TOKENS = int(np.percentile(prompt_token_lens, 10))
PROMPT_P95_TOKENS = int(np.percentile(prompt_token_lens, 95))

# 128-word continuation token budget with 35% fragmentation headroom.
raw_cont_budget = int(math.ceil(128 * tpw * 1.35))
EXPECTED_CONT_TOKENS = int(math.ceil(raw_cont_budget / 64) * 64)

SEQ_LEN = int(math.ceil((PROMPT_P95_TOKENS + EXPECTED_CONT_TOKENS) / 128) * 128)

if SEQ_LEN > MAX_POSITIONS:
    raise RuntimeError(
        f"Measured SEQ_LEN={SEQ_LEN} exceeds MAX_POSITIONS={MAX_POSITIONS}. "
        "Increase MAX_POSITIONS before preprocessing/training."
    )

# Prompt-like split positions used only to choose where continuation loss begins.
# Attention remains causal on BOTH sides of the split.
PROMPT_SPLIT_MIN = max(
    64,
    min(PROMPT_P10_TOKENS, SEQ_LEN - EXPECTED_CONT_TOKENS)
)
PROMPT_SPLIT_MAX = max(
    PROMPT_SPLIT_MIN,
    min(PROMPT_P95_TOKENS, SEQ_LEN - EXPECTED_CONT_TOKENS)
)

PROMPT_KEEP_TOKENS = min(
    PROMPT_P95_TOKENS,
    SEQ_LEN - EXPECTED_CONT_TOKENS
)

print("Official prompts:", len(prompt_token_lens))
print(
    "BPE tokens min/mean/p10/p95/max:",
    min(prompt_token_lens),
    f"{np.mean(prompt_token_lens):.0f}",
    PROMPT_P10_TOKENS,
    PROMPT_P95_TOKENS,
    max(prompt_token_lens),
)
print("Tokens/word:", round(tpw, 3))
print("Expected 128-word continuation budget:", EXPECTED_CONT_TOKENS, "BPE tokens")
print("CHOSEN SEQ_LEN:", SEQ_LEN)
print("Continuation-loss split range:", PROMPT_SPLIT_MIN, "to", PROMPT_SPLIT_MAX)
print("Inference prompt keep:", PROMPT_KEEP_TOKENS, "BPE tokens")
print("MAX_POSITIONS:", MAX_POSITIONS)

assert PROMPT_SPLIT_MAX + EXPECTED_CONT_TOKENS <= SEQ_LEN
assert PROMPT_KEEP_TOKENS + EXPECTED_CONT_TOKENS <= SEQ_LEN


Official test file: Phase_1_Pretraining/Test/test.json


test.json: 0.00B [00:00, ?B/s]

Official prompts: 1000
BPE tokens min/mean/p10/p95/max: 1030 1082 1050 1139 1367
Tokens/word: 2.114
Expected 128-word continuation budget: 384 BPE tokens
CHOSEN SEQ_LEN: 1536
Continuation-loss split range: 1050 to 1139
Inference prompt keep: 1139 BPE tokens
MAX_POSITIONS: 2560


In [7]:
# Cell 7 — Stream/tokenize ALL FOUR sources to uint16 train/val binaries (400M train target)
# This is the CPU-heavy preprocessing stage. It is resumable at the FILE level:
# if meta.json + both bins already exist, this entire stage is skipped.

_WORKER_TOKENIZER = None
_WORKER_EOS = None

def _encode_worker(text):
    ids = _WORKER_TOKENIZER.encode(text)
    ids.append(_WORKER_EOS)
    return np.asarray(ids, dtype=np.uint16)

def make_pool():
    global _WORKER_TOKENIZER, _WORKER_EOS
    _WORKER_TOKENIZER = tokenizer
    _WORKER_EOS = tokenizer.eos_token_id

    if ENCODE_PROCESSES <= 1:
        return None

    import multiprocessing as mp
    try:
        ctx = mp.get_context("fork")
        return ctx.Pool(ENCODE_PROCESSES)
    except Exception as e:
        print("Multiprocessing unavailable; falling back to single process:", repr(e))
        return None

def encode_many(texts, pool):
    if pool is None:
        return [_encode_worker(x) for x in texts]
    return list(pool.imap(_encode_worker, texts, chunksize=8))

if META_PATH.exists() and TRAIN_BIN.exists() and VAL_BIN.exists() and VAL_GEN_PATH.exists() and VAL_REF_PATH.exists():
    meta = json.load(open(META_PATH))
    print("Existing tokenized corpus found; skipping preprocessing.")
    print(json.dumps(meta, indent=2)[:2500])
else:
    # Start fresh so partial old bins cannot be mistaken for complete data.
    for p in [TRAIN_BIN, VAL_BIN, VAL_GEN_PATH, VAL_REF_PATH, META_PATH]:
        if p.exists():
            p.unlink()

    f_train = open(TRAIN_BIN, "wb")
    f_val = open(VAL_BIN, "wb")
    pool = make_pool()

    meta = {
        "vocab_size":len(tokenizer),
        "seq_len":SEQ_LEN,
        "prompt_p10_tokens":PROMPT_P10_TOKENS,
        "prompt_p95_tokens":PROMPT_P95_TOKENS,
        "expected_cont_tokens":EXPECTED_CONT_TOKENS,
        "prompt_split_min":PROMPT_SPLIT_MIN,
        "prompt_split_max":PROMPT_SPLIT_MAX,
        "sources":{},
        "target_total_tokens":TARGET_TOTAL_TOKENS,
    }
    val_gen = {}
    val_ref = {}

    rng = random.Random(SEED + 99)

    try:
        for key, target in TRAIN_TOKEN_BUDGET.items():
            path = fetch_source(key)

            # Estimate tokens/char from the first ~200 documents.
            probe_docs = []
            probe_chars = 0
            for doc in iter_docs(path):
                probe_docs.append(doc)
                probe_chars += len(doc)
                if len(probe_docs) >= 200:
                    break

            probe_encoded = encode_many(probe_docs, pool)
            probe_tokens = sum(len(x) for x in probe_encoded)
            tokens_per_char = probe_tokens / max(1, probe_chars)
            est_tokens = census[key]["est_chars"] * tokens_per_char

            # Oversample ~8%, then stop exactly when the requested token budget is reached.
            q_train = min(1.0, 1.08 * target / max(1.0, est_tokens))
            q_val = min(
                1.0,
                1.20 * VAL_TOKENS_PER_SOURCE /
                max(1.0, est_tokens * VAL_PERMILLE / 1000.0)
            )

            print(
                f"\n[{key}] est_tokens≈{est_tokens/1e6:.0f}M | "
                f"target={target/1e6:.0f}M | q_train={q_train:.5f} | q_val={q_val:.5f}",
                flush=True
            )

            n_train = 0
            n_val = 0
            docs_seen = 0
            gen_found = 0
            t0 = time.time()

            batch_texts = []
            batch_is_val = []

            def flush_batch():
                global n_train, n_val, batch_texts, batch_is_val
                if not batch_texts:
                    return

                arrays = encode_many(batch_texts, pool)

                for arr, is_val in zip(arrays, batch_is_val):
                    if is_val:
                        if n_val >= VAL_TOKENS_PER_SOURCE:
                            continue
                        remaining = VAL_TOKENS_PER_SOURCE - n_val
                        a = arr[:remaining]
                        a.tofile(f_val)
                        n_val += len(a)
                    else:
                        if n_train >= target:
                            continue
                        remaining = target - n_train
                        a = arr[:remaining]
                        a.tofile(f_train)
                        n_train += len(a)

                batch_texts = []
                batch_is_val = []

            for idx, doc in enumerate(iter_docs(path)):
                docs_seen += 1
                if len(doc) < 200:
                    continue

                # Deterministic document-level heldout split.
                is_val_doc = ((idx * 2654435761) % 1000) < VAL_PERMILLE

                if is_val_doc:
                    if n_val >= VAL_TOKENS_PER_SOURCE:
                        continue
                    if rng.random() > q_val:
                        continue

                    words = doc.split()
                    if gen_found < VAL_GEN_PER_SOURCE and len(words) >= 640:
                        st = rng.randint(0, len(words)-640)
                        exid = f"{key}_{gen_found:02d}"
                        val_gen[exid] = {"text":" ".join(words[st:st+512])}
                        val_ref[exid] = {"text":" ".join(words[st+512:st+640])}
                        gen_found += 1

                    batch_texts.append(doc)
                    batch_is_val.append(True)

                else:
                    if n_train >= target:
                        # Keep scanning only until validation is also sufficiently filled.
                        if n_val >= int(0.95 * VAL_TOKENS_PER_SOURCE):
                            break
                        continue
                    if rng.random() > q_train:
                        continue
                    batch_texts.append(doc)
                    batch_is_val.append(False)

                if len(batch_texts) >= 64:
                    flush_batch()

                    if n_train and n_train % 25_000_000 < 100_000:
                        print(
                            f"  [{key}] train={n_train/1e6:.1f}M "
                            f"val={n_val/1e6:.2f}M docs_seen={docs_seen:,} "
                            f"elapsed={(time.time()-t0)/60:.1f}m",
                            flush=True
                        )

            flush_batch()

            meta["sources"][key] = {
                "train_tokens":n_train,
                "val_tokens":n_val,
                "docs_seen":docs_seen,
                "tokens_per_char_probe":tokens_per_char,
                "q_train":q_train,
            }

            print(
                f"[{key}] DONE train={n_train/1e6:.1f}M "
                f"val={n_val/1e6:.2f}M local_gen={gen_found} "
                f"time={(time.time()-t0)/60:.1f}m"
            )

            maybe_delete_download(path)

    finally:
        if pool is not None:
            pool.close()
            pool.join()
        f_train.close()
        f_val.close()

    json.dump(val_gen, open(VAL_GEN_PATH,"w"), ensure_ascii=False, indent=2)
    json.dump(val_ref, open(VAL_REF_PATH,"w"), ensure_ascii=False, indent=2)

    meta["actual_train_tokens"] = os.path.getsize(TRAIN_BIN)//2
    meta["actual_val_tokens"] = os.path.getsize(VAL_BIN)//2
    json.dump(meta, open(META_PATH,"w"), indent=2)

    print("\nPreprocessing complete.")
    print("train tokens:", f"{meta['actual_train_tokens']/1e6:.1f}M")
    print("val tokens:", f"{meta['actual_val_tokens']/1e6:.2f}M")
    print("binary train size:", f"{os.path.getsize(TRAIN_BIN)/1e9:.2f} GB")
    print("local generation examples:", len(val_gen))


[sc] est_tokens≈474M | target=144M | q_train=0.32836 | q_val=0.25336
  [sc] train=25.1M val=0.10M docs_seen=8,623 elapsed=0.2m
  [sc] train=50.0M val=0.22M docs_seen=17,251 elapsed=0.4m
[sc] DONE train=144.0M val=0.50M local_gen=40 time=1.2m
Deleted cached source to save disk: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/blobs/96d8e4d106292e7d5b6c4cb81e125f7e8cdee50b66302a373f547178b5364923

[hc] est_tokens≈3652M | target=144M | q_train=0.04258 | q_val=0.03286
  [hc] train=0.0M val=0.00M docs_seen=1,523 elapsed=0.0m
  [hc] train=25.0M val=0.13M docs_seen=585,195 elapsed=0.4m
  [hc] train=75.1M val=0.26M docs_seen=1,102,593 elapsed=1.0m
  [hc] train=100.0M val=0.32M docs_seen=1,604,216 elapsed=1.4m
  [hc] train=100.1M val=0.32M docs_seen=1,605,911 elapsed=1.4m
[hc] DONE train=144.0M val=0.46M local_gen=40 time=2.1m
Deleted cached source to save disk: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/blobs/59b3d4df6b492dd03d89197aa634b583

In [8]:
# Cell 8 — Memory-mapped block dataset sanity
meta = json.load(open(META_PATH))
train_mm = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_mm = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

BLOCK_TOKENS = SEQ_LEN + 1
N_TRAIN_BLOCKS = len(train_mm) // BLOCK_TOKENS
N_VAL_BLOCKS = len(val_mm) // BLOCK_TOKENS

print("Train token IDs:", f"{len(train_mm):,}")
print("Val token IDs:", f"{len(val_mm):,}")
print("Train blocks:", f"{N_TRAIN_BLOCKS:,}")
print("Val blocks:", f"{N_VAL_BLOCKS:,}")

assert N_TRAIN_BLOCKS > 0 and N_VAL_BLOCKS > 0
assert len(tokenizer) < 65536

sample = np.asarray(train_mm[:BLOCK_TOKENS], dtype=np.int64)
print("Sample block shape:", sample.shape)
print("Decoded start:", repr(tokenizer.decode(sample[:80])[:300]))

print("Requested training-token budget:", f"{TARGET_TOTAL_TOKENS:,}")
assert len(train_mm) >= TARGET_TOTAL_TOKENS, (
    f"Training binary too small: {len(train_mm):,} < {TARGET_TOTAL_TOKENS:,}"
)
print("✅ 400M corpus sanity passed")


Train token IDs: 400,000,000
Val token IDs: 1,959,805
Train blocks: 260,247
Val blocks: 1,275
Sample block shape: (1537,)
Decoded start: 'JUDGMENT\n\nK.C. Das Gupta, J.\n\n1. The only question that arises for consideration in this appeal by the State Bank of India is as regards the effect of Para 292 of the Bank Award, as modified, on the respondent, '
Requested training-token budget: 400,000,000
✅ 400M corpus sanity passed


In [13]:
# Cell 9 — ~99.31M Efficient Modern Decoder: GQA + RoPE + RMSNorm + SwiGLU   (v2)
# Same parameters / same state_dict keys / same maths as v1 (v1 checkpoints load unchanged). What changed:
#   * RoPE cos/sin are gathered ONCE per forward (v1: once per layer = 34 gathers + 17 GPU->CPU syncs per step)
#   * decode steps use the GQA-native path: the 5 query heads of a KV group become 5 query rows, so the cached
#     K/V are never expanded 5x (v1 copied the whole expanded cache every layer at every generated token)
#   * multi-token chunks that follow a cache are now causal (v1 silently made them bidirectional; unused so far)
#   * build_model() so training always starts from fresh weights; embeddings excluded from weight decay in Cell 11

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32.to(dtype) * self.weight)


class RotaryTable(nn.Module):
    """cos/sin table (non-persistent buffers -> nothing is added to the state_dict)."""
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
        self.max_positions = max_positions

    def forward(self, position_ids):
        # (B,T) -> two (B,1,T,D/2) tensors, shared by every layer
        return (self.cos_cached[position_ids].unsqueeze(1),
                self.sin_cached[position_ids].unsqueeze(1))


def apply_rope(x, cos, sin):
    # x: (B,H,T,D) with interleaved (even, odd) rotation pairs -- identical convention to v1
    B, H, T, D = x.shape
    cos = cos.to(x.dtype)
    sin = sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)


class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH = True     # flip to False to fall back to the (slower) expand-K/V path; used by the self-test

    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        new_kv = (k, v) if use_cache else None          # compact cache: only KV heads are stored
        S = k.size(2)
        drop = self.dropout if self.training else 0.0

        # ---------------- single new token (cached decoding): GQA-native, no K/V expansion
        if T == 1 and past_len > 0 and self.GQA_DECODE_PATH:
            qg = q.reshape(B, KV, self.kv_repeat, D)     # head h = kv*rep + r  ->  row r of group kv
            mask = None if key_valid is None else key_valid[:, None, None, :]
            y = F.scaled_dot_product_attention(qg, k, v, attn_mask=mask, dropout_p=drop, is_causal=False)
            y = y.reshape(B, H, 1, D).transpose(1, 2).contiguous().view(B, 1, C)
            return self.o_proj(y), new_kv

        # ---------------- general path (training / prefill / chunks)
        if self.kv_repeat != 1:
            k_attn = k.repeat_interleave(self.kv_repeat, dim=1)
            v_attn = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            k_attn, v_attn = k, v

        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True                             # fast path used by training
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:                        # left-padding queries keep one safe key (themselves)
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]

        y = F.scaled_dot_product_attention(
            q, k_attn, v_attn, attn_mask=attn_mask, dropout_p=drop, is_causal=is_causal
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.norm1(x), cos, sin, past_kv=past_kv, use_cache=use_cache, key_valid=key_valid
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class EfficientDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads, n_kv_heads,
        ffn_dim, dropout, max_positions, rope_base, rms_eps
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(d_model // n_heads, max_positions, rope_base)

        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps)
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model, rms_eps)

        self.apply(self._init_weights)
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.o_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff.down_proj.weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(self, input_ids, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len + T > self.max_positions:
            raise ValueError(f"context {past_len + T} exceeds max_positions={self.max_positions}")

        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device)[None, :].expand(B, T)
        cos, sin = self.rope(position_ids)               # once for all layers (callers keep ids < max_positions)

        x = self.drop(self.token_emb(input_ids))
        new_cache = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, cos, sin, past_kv=past, use_cache=use_cache, key_valid=key_valid)
            if use_cache:
                new_cache.append(kv)
        return self.final_norm(x), new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)   # tied token/output embedding


def build_model():
    return EfficientDecoderLM(
        vocab_size=len(tokenizer), d_model=D_MODEL, n_layers=N_LAYERS,
        n_heads=N_HEADS, n_kv_heads=N_KV_HEADS, ffn_dim=FFN_DIM, dropout=DROPOUT,
        max_positions=MAX_POSITIONS, rope_base=ROPE_BASE, rms_eps=RMS_EPS,
    ).to(DEVICE)


model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")
print(
    f"Architecture: {N_LAYERS} layers | d={D_MODEL} | "
    f"{N_HEADS}Q/{N_KV_HEADS}KV GQA | RoPE | RMSNorm | SwiGLU"
)
assert 95_000_000 <= n_params <= 105_000_000, n_params


Model parameters: 99.31M
Architecture: 17 layers | d=640 | 10Q/2KV GQA | RoPE | RMSNorm | SwiGLU


In [14]:
# Cell 9b — Architecture correctness self-test before spending hours training   (v2: much stricter)
# Checks: finite outputs, causality, cached == full decoding, left-padded batched decoding == un-padded single
# decoding (this is exactly how generation runs), chunked prefill, GQA-native decode == expanded decode, compact cache.

@torch.no_grad()
def run_self_test(m, atol=2e-3):
    was_training = m.training
    tf32 = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False
    m.eval()
    dev = next(m.parameters()).device
    V = m.vocab_size
    g = torch.Generator().manual_seed(0)
    rnd = lambda *s: torch.randint(4, V, s, generator=g).to(dev)
    maxdiff = lambda a, b: (a.float() - b.float()).abs().max().item()

    # 1) finite forward
    B, T = 2, 24
    x = rnd(B, T)
    h_full, _ = m.forward_hidden(x)
    assert torch.isfinite(h_full).all(), "non-finite hidden states"

    # 2) causality
    x2 = x.clone(); x2[:, -1] = 4 + (x2[:, -1] + 17) % (V - 4)
    h2, _ = m.forward_hidden(x2)
    assert maxdiff(h_full[:, :-1], h2[:, :-1]) < atol, "causality violated"

    # 3) prefill T-1 + one cached token == full forward
    _, cache = m.forward_hidden(x[:, :-1], use_cache=True)
    pos = torch.full((B, 1), T - 1, dtype=torch.long, device=dev)
    last, cache2 = m.forward_hidden(x[:, -1:], past_key_values=cache, use_cache=True, position_ids=pos)
    assert maxdiff(last[:, -1], h_full[:, -1]) < atol, "KV-cache mismatch"

    # 4) compact GQA cache
    k0, v0 = cache2[0]
    assert k0.shape[1] == N_KV_HEADS and v0.shape[1] == N_KV_HEADS and k0.shape[2] == T

    # 5) chunked prefill: 3 tokens appended to a cache must be causal among themselves
    _, c_a = m.forward_hidden(x[:, :15], use_cache=True)
    h_chunk, _ = m.forward_hidden(x[:, 15:18], past_key_values=c_a, use_cache=True)
    assert maxdiff(h_chunk, h_full[:, 15:18]) < atol, "chunked cached forward is not causal / not equal"

    # 6) left-padded BATCH generation path == each prompt run alone without padding
    lens = [24, 17, 9]
    Bp, S = len(lens), max(lens)
    seqs = [rnd(L) for L in lens]
    ids = torch.zeros(Bp, S, dtype=torch.long, device=dev)
    valid = torch.zeros(Bp, S, dtype=torch.bool, device=dev)
    for b, e in enumerate(seqs):
        ids[b, S - len(e):] = e
        valid[b, S - len(e):] = True
    pos = (valid.long().cumsum(1) - 1).clamp(min=0)
    nxt = valid.sum(1)
    new_tok = [rnd(Bp, 1) for _ in range(5)]

    def decode_batched(use_gqa_path):
        GQACausalSelfAttention.GQA_DECODE_PATH = use_gqa_path
        h, c = m.forward_hidden(ids, use_cache=True, position_ids=pos, key_valid=valid)
        outs, v, n = [h[:, -1]], valid, nxt.clone()
        for t in new_tok:
            v = torch.cat([v, torch.ones(Bp, 1, dtype=torch.bool, device=dev)], 1)
            h, c = m.forward_hidden(t, past_key_values=c, use_cache=True, position_ids=n[:, None], key_valid=v)
            n = n + 1
            outs.append(h[:, -1])
        return torch.stack(outs, 1)                                  # (Bp, 6, C)

    out_gqa = decode_batched(True)
    out_exp = decode_batched(False)
    GQACausalSelfAttention.GQA_DECODE_PATH = True
    assert maxdiff(out_gqa, out_exp) < atol, "GQA-native decode differs from expanded decode"

    for b, e in enumerate(seqs):
        h, c = m.forward_hidden(e[None, :], use_cache=True)
        ref, n = [h[:, -1]], len(e)
        for t in new_tok:
            h, c = m.forward_hidden(t[b:b + 1], past_key_values=c, use_cache=True,
                                    position_ids=torch.tensor([[n]], device=dev))
            n += 1
            ref.append(h[:, -1])
        ref = torch.stack(ref, 1)[0]
        assert maxdiff(out_gqa[b], ref) < atol, f"left-padded batch row {b} differs from the un-padded run"

    # 7) tied output projection
    logits = m.project(h_full[:, -1:])
    assert logits.shape == (B, 1, V) and torch.isfinite(logits).all()

    torch.backends.cuda.matmul.allow_tf32 = tf32
    m.train(was_training)
    print("SELF-TEST PASSED:")
    print("  finite forward / causality / cached == full decoding: OK")
    print("  chunked cached forward is causal: OK")
    print("  left-padded batched generation path == un-padded single run: OK")
    print("  GQA-native decode == expanded decode; compact cache "
          f"({N_KV_HEADS} KV heads for {N_HEADS} query heads): OK")

run_self_test(model)


SELF-TEST PASSED:
  finite forward / causality / cached == full decoding: OK
  chunked cached forward is causal: OK
  left-padded batched generation path == un-padded single run: OK
  GQA-native decode == expanded decode; compact cache (2 KV heads for 10 query heads): OK


In [15]:
# Cell 10 — Pilot/full run setup
# Start with pilot. For the full run:
#   1) stop pilot after ~100–150 steps,
#   2) change RUN_MODE below to "full",
#   3) rerun this cell and Cell 11   (v2: Cell 11 builds fresh random weights itself; no need to rerun Cell 9).

RUN_MODE = "full"   # <- change to "full" only after a healthy pilot

if RUN_MODE == "pilot":
    RUN_NAME = "eff_pilot"
    TRAIN_BLOCK_LIMIT = 5_000_000 // SEQ_LEN
elif RUN_MODE == "full":
    RUN_NAME = "eff_full"
    TRAIN_BLOCK_LIMIT = None
else:
    raise ValueError("RUN_MODE must be 'pilot' or 'full'")

CKPT_DIR = ROOT / "checkpoints" / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)

LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT = CKPT_DIR / "best.pt"
FINAL_CKPT = CKPT_DIR / "final.pt"

OUTSIDE_LATEST_CKPT = Path(f"/kaggle/working/{RUN_NAME}_latest.pt")
OUTSIDE_BEST_CKPT = Path(f"/kaggle/working/{RUN_NAME}_best.pt")

print("RUN_MODE:", RUN_MODE)
print("RUN_NAME:", RUN_NAME)
print("TRAIN_BLOCK_LIMIT:", TRAIN_BLOCK_LIMIT)
if TRAIN_BLOCK_LIMIT is not None:
    print("Approx pilot raw tokens:", TRAIN_BLOCK_LIMIT * SEQ_LEN)
print("Checkpoint dir:", CKPT_DIR)
print("Latest copy:", OUTSIDE_LATEST_CKPT)
print("Best copy:", OUTSIDE_BEST_CKPT)


RUN_MODE: full
RUN_NAME: eff_full
TRAIN_BLOCK_LIMIT: None
Checkpoint dir: /kaggle/working/cs781_effdecoder_100m/checkpoints/eff_full
Latest copy: /kaggle/working/eff_full_latest.pt
Best copy: /kaggle/working/eff_full_best.pt


In [16]:
# Cell 11 — Efficient-decoder training loop: one shuffled pass over ~400M tokens, resumable   (v2)
# v2: builds FRESH weights itself (then resumes from latest.pt if present) -> no 'rerun Cell 9' step;
#     no weight decay on the tied embedding; loss accumulated on-GPU (one host sync per optimizer step).
def get_block(mm, block_index):
    start = int(block_index) * BLOCK_TOKENS
    arr = np.asarray(mm[start:start + BLOCK_TOKENS], dtype=np.int64)
    return torch.from_numpy(arr)

def get_batch_from_indices(mm, indices):
    return torch.stack([get_block(mm, i) for i in indices], dim=0).to(
        DEVICE, non_blocking=True
    )

# Per-source validation boundaries.
_val_bounds, _cum = [], 0
for _k, _s in meta["sources"].items():
    _val_bounds.append((_k, _cum, _cum + int(_s["val_tokens"])))
    _cum += int(_s["val_tokens"])

def val_source_of_block(idx):
    start = int(idx) * BLOCK_TOKENS
    for k, lo, hi in _val_bounds:
        if lo <= start < hi:
            return k
    return "other"

LAST_VAL_DETAIL = {}

@torch.no_grad()
def evaluate_val(model, max_blocks=VAL_EVAL_BLOCKS):
    """
    Decoder attention is causal everywhere.
    Best-checkpoint selection uses prompt-like continuation loss (closest to Phase-1 test).
    Full causal-LM loss is logged as an additional health metric.
    """
    model.eval()

    n = min(N_VAL_BLOCKS, max_blocks)
    if n <= 0:
        return float("inf")

    ids = np.linspace(0, N_VAL_BLOCKS - 1, n, dtype=np.int64)

    cont_losses = []
    full_losses = []
    by_src = {}

    p = max(1, min(SEQ_LEN, PROMPT_KEEP_TOKENS))
    start = max(0, p - 1)

    for idx in ids:
        batch = get_batch_from_indices(val_mm, [idx])
        x = batch[:, :-1]
        y = batch[:, 1:]

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            h, _ = model.forward_hidden(x)

            # Task-aligned continuation validation loss.
            cont_logits = model.project(h[:, start:, :])
            cont_targets = y[:, start:]
            cont_loss = F.cross_entropy(
                cont_logits.reshape(-1, cont_logits.size(-1)),
                cont_targets.reshape(-1),
            )

            # A small sampled full-LM health metric is computed below to avoid
            # doubling the large vocab projection on every validation block.

        cv = float(cont_loss.item())
        cont_losses.append(cv)
        by_src.setdefault(val_source_of_block(idx), []).append(cv)

    # Full causal-LM health check on <=32 evenly spaced validation blocks.
    full_ids = ids[::max(1, len(ids) // 32)][:32]
    for idx in full_ids:
        batch = get_batch_from_indices(val_mm, [idx])
        x = batch[:, :-1]
        y = batch[:, 1:]
        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            h, _ = model.forward_hidden(x)
            logits = model.project(h)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )
        full_losses.append(float(loss.item()))

    LAST_VAL_DETAIL.clear()
    LAST_VAL_DETAIL["val_cont_by_source"] = {
        k: sum(v) / len(v) for k, v in by_src.items()
    }
    LAST_VAL_DETAIL["val_full_causal"] = (
        sum(full_losses) / len(full_losses) if full_losses else None
    )

    model.train()
    return sum(cont_losses) / len(cont_losses)


# Always start from freshly initialised weights (a checkpoint, if any, is loaded further below).
model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Fresh model built: {n_params/1e6:.2f}M parameters")

# AdamW: weight decay on 2-D weight matrices only (not on the tied embedding, not on RMSNorm gains).
def _decays(n_, p):
    return p.requires_grad and p.ndim >= 2 and "token_emb" not in n_
decay_params = [p for n_, p in model.named_parameters() if _decays(n_, p)]
nodecay_params = [p for n_, p in model.named_parameters() if p.requires_grad and not _decays(n_, p)]

param_groups = [
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": nodecay_params, "weight_decay": 0.0},
]

optim_kwargs = dict(
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
)

try:
    optimizer = torch.optim.AdamW(
        param_groups,
        fused=(DEVICE.type == "cuda"),
        **optim_kwargs,
    )
    print("AdamW fused:", DEVICE.type == "cuda")
except Exception:
    optimizer = torch.optim.AdamW(param_groups, **optim_kwargs)
    print("Using standard AdamW.")

# A reused Architecture-1 binary may contain ~500M tokens.
# Never let a normal efficient-decoder full run silently exceed this notebook's 400M budget.
AUTO_FULL_BLOCK_LIMIT = max(1, TARGET_TOTAL_TOKENS // SEQ_LEN)

N_TRAIN_USED = (
    min(N_TRAIN_BLOCKS, AUTO_FULL_BLOCK_LIMIT)
    if TRAIN_BLOCK_LIMIT is None
    else min(N_TRAIN_BLOCKS, int(TRAIN_BLOCK_LIMIT))
)

print(
    f"Training on {N_TRAIN_USED:,} of {N_TRAIN_BLOCKS:,} blocks "
    f"(~{N_TRAIN_USED * SEQ_LEN / 1e6:.0f}M tokens) | run={RUN_NAME}"
)

micro_steps_total = math.ceil(N_TRAIN_USED / MICRO_BATCH_SIZE)
opt_steps_total = math.ceil(micro_steps_total / GRAD_ACCUM_STEPS)
warmup_steps = max(1, int(opt_steps_total * WARMUP_RATIO))

def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-6, step / max(1, warmup_steps))

    if LR_SCHEDULE == "cosine":
        progress = (step - warmup_steps) / max(1, opt_steps_total - warmup_steps)
        cosine = 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
        return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * cosine

    decay_start = int(opt_steps_total * (1.0 - WSD_DECAY_FRAC))
    if step < decay_start:
        return 1.0

    prog = (step - decay_start) / max(1, opt_steps_total - decay_start)
    return 1.0 + (MIN_LR_RATIO - 1.0) * min(1.0, prog)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# Deterministic order enables exact cursor-based resume.
perm_gen = torch.Generator()
perm_gen.manual_seed(SEED + 777)
block_perm = torch.randperm(N_TRAIN_BLOCKS, generator=perm_gen).numpy()

cursor = 0
global_step = 0
best_val = float("inf")

def try_restore_from_hf():
    if LATEST_CKPT.exists() or not HF_CHECKPOINT_REPO:
        return
    try:
        p = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO,
            filename=f"{RUN_NAME}/latest.pt",
            repo_type="model",
            token=HF_TOKEN,
        )
        shutil.copy2(p, LATEST_CKPT)
        try:
            os.remove(os.path.realpath(p))
        except Exception:
            pass
        print("Restored latest.pt from HF checkpoint repo.")
    except Exception as e:
        print("No restorable checkpoint on HF (starting fresh):", repr(e)[:160])

try_restore_from_hf()

if LATEST_CKPT.exists():
    ckpt = torch.load(LATEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    cursor = int(ckpt["cursor"])
    global_step = int(ckpt["global_step"])
    best_val = float(ckpt["best_val"])
    if "python_random_state" in ckpt:
        random.setstate(ckpt["python_random_state"])
    print(
        f"RESUME: cursor={cursor:,}/{N_TRAIN_USED:,} | "
        f"step={global_step:,} | best_val={best_val:.4f}"
    )

def atomic_torch_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def atomic_copy(src, dst):
    """Copy a completed checkpoint to an easy-to-download location atomically."""
    src, dst = Path(src), Path(dst)
    tmp = Path(str(dst) + ".tmp")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

def checkpoint_object():
    return {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "cursor": cursor,
        "global_step": global_step,
        "best_val": best_val,
        "python_random_state": random.getstate(),
        "config": {
            "architecture": "efficient_gqa_rope_swiglu_decoder",
            "vocab_size": len(tokenizer),
            "d_model": D_MODEL,
            "n_layers": N_LAYERS,
            "n_heads": N_HEADS,
            "n_kv_heads": N_KV_HEADS,
            "ffn_dim": FFN_DIM,
            "rope_base": ROPE_BASE,
            "rms_eps": RMS_EPS,
            "dropout": DROPOUT,
            "max_positions": MAX_POSITIONS,
            "seq_len": SEQ_LEN,
            "prompt_p10_tokens": PROMPT_P10_TOKENS,
            "prompt_p95_tokens": PROMPT_P95_TOKENS,
            "expected_cont_tokens": EXPECTED_CONT_TOKENS,
            "prompt_split_min": PROMPT_SPLIT_MIN,
            "prompt_split_max": PROMPT_SPLIT_MAX,
            "loss_on_all_positions": LOSS_ON_ALL_POSITIONS,
        },
    }

_hf_upload_count = 0

def maybe_upload_latest_to_hf():
    global _hf_upload_count
    if not HF_CHECKPOINT_REPO:
        return

    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(
            HF_CHECKPOINT_REPO,
            repo_type="model",
            private=True,
            exist_ok=True,
        )
        api.upload_file(
            path_or_fileobj=str(LATEST_CKPT),
            path_in_repo=f"{RUN_NAME}/latest.pt",
            repo_id=HF_CHECKPOINT_REPO,
            repo_type="model",
            commit_message=f"{RUN_NAME} checkpoint step {global_step}",
        )
        _hf_upload_count += 1
        print(
            f"Uploaded {RUN_NAME}/latest.pt to private HF checkpoint repo "
            f"(upload #{_hf_upload_count})."
        )

        if (
            HF_SQUASH_EVERY_N_UPLOADS
            and _hf_upload_count % HF_SQUASH_EVERY_N_UPLOADS == 0
        ):
            try:
                api.super_squash_history(
                    repo_id=HF_CHECKPOINT_REPO,
                    repo_type="model",
                )
                print("Squashed HF repo history.")
            except Exception as e:
                print("History squash skipped:", repr(e)[:120])

    except Exception as e:
        print("HF checkpoint upload FAILED (training continues):", repr(e)[:200])

LOG_JSONL = CKPT_DIR / "train_log.jsonl"

def append_log(rec):
    rec = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), **rec}
    with open(LOG_JSONL, "a") as f:
        f.write(json.dumps(rec) + "\n")

def save_latest_and_expose():
    atomic_torch_save(checkpoint_object(), LATEST_CKPT)
    atomic_copy(LATEST_CKPT, OUTSIDE_LATEST_CKPT)
    print("Saved latest checkpoint + refreshed:", OUTSIDE_LATEST_CKPT)

_CFG_NAMES = [
    "RUN_NAME", "VOCAB_SIZE", "SEQ_LEN", "MAX_POSITIONS",
    "D_MODEL", "N_LAYERS", "N_HEADS", "N_KV_HEADS", "FFN_DIM", "ROPE_BASE", "RMS_EPS", "DROPOUT",
    "PROMPT_SPLIT_MIN", "PROMPT_SPLIT_MAX", "PROMPT_KEEP_TOKENS",
    "EXPECTED_CONT_TOKENS", "MICRO_BATCH_SIZE", "GRAD_ACCUM_STEPS",
    "LEARNING_RATE", "MIN_LR_RATIO", "WARMUP_RATIO", "WEIGHT_DECAY",
    "GRAD_CLIP", "LR_SCHEDULE", "WSD_DECAY_FRAC",
    "TRAIN_BLOCK_LIMIT", "LOSS_ON_ALL_POSITIONS", "SEED",
]

run_config = {k: globals()[k] for k in _CFG_NAMES}
run_config.update(
    architecture="efficient_gqa_rope_swiglu_decoder",
    n_params=n_params,
    n_train_blocks_used=int(N_TRAIN_USED),
    opt_steps_total=int(opt_steps_total),
    resumed_at_step=int(global_step),
)
json.dump(run_config, open(CKPT_DIR / "run_config.json", "w"), indent=2)

model.train()
optimizer.zero_grad(set_to_none=True)

running_loss = 0.0
running_updates = 0
train_start = time.time()
last_report_time = train_start
last_report_cursor = cursor

while cursor < N_TRAIN_USED:
    accum_loss = torch.zeros((), device=DEVICE)
    accum_count = 0

    for _ in range(GRAD_ACCUM_STEPS):
        if cursor >= N_TRAIN_USED:
            break

        inds = block_perm[cursor:min(cursor + MICRO_BATCH_SIZE, N_TRAIN_USED)]
        if len(inds) == 0:
            break

        batch = get_batch_from_indices(train_mm, inds)
        cursor += len(inds)

        x = batch[:, :-1]
        y = batch[:, 1:]

        if LOSS_ON_ALL_POSITIONS:
            # Standard decoder-only causal language modelling:
            # every hidden state predicts the next token.
            start = 0
        else:
            # Optional task-focused ablation: only suffix positions contribute to loss.
            # The attention pattern remains causal everywhere.
            p = random.randint(PROMPT_SPLIT_MIN, PROMPT_SPLIT_MAX)
            p = max(1, min(x.size(1), p))
            start = max(0, p - 1)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            h, _ = model.forward_hidden(x)
            logits = model.project(h[:, start:, :])
            targets = y[:, start:]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()

        accum_loss += loss.detach().float()
        accum_count += 1

    if accum_count == 0:
        break

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

    global_step += 1
    running_loss += float(accum_loss.item()) / accum_count
    running_updates += 1

    if global_step % LOG_EVERY == 0:
        now = time.time()
        blocks_done = cursor - last_report_cursor
        tokens_done = blocks_done * SEQ_LEN
        tok_s = tokens_done / max(1e-6, now - last_report_time)

        remaining_tokens = max(0, N_TRAIN_USED - cursor) * SEQ_LEN
        eta_h = remaining_tokens / max(tok_s, 1e-9) / 3600

        avg = running_loss / max(1, running_updates)
        lr = optimizer.param_groups[0]["lr"]

        print(
            f"step={global_step:,}/{opt_steps_total:,} | "
            f"blocks={cursor:,}/{N_TRAIN_USED:,} | "
            f"processed≈{cursor * SEQ_LEN / 1e6:.1f}M tok | "
            f"loss={avg:.4f} | lr={lr:.2e} | "
            f"{tok_s:,.0f} tok/s | ETA≈{eta_h:.1f}h",
            flush=True,
        )

        append_log({
            "type": "train",
            "step": global_step,
            "tokens": int(cursor * SEQ_LEN),
            "loss": avg,
            "lr": lr,
            "tok_s": tok_s,
        })

        running_loss = 0.0
        running_updates = 0
        last_report_time = now
        last_report_cursor = cursor

    if global_step % VAL_EVERY == 0 or cursor >= N_TRAIN_USED:
        val_loss = evaluate_val(model)
        ppl = math.exp(min(20, val_loss))

        print(
            f"VALIDATION step={global_step:,}: "
            f"continuation_loss={val_loss:.4f} ppl={ppl:.2f}"
        )
        print(
            "  per-source continuation val loss:",
            {
                k: round(v, 4)
                for k, v in LAST_VAL_DETAIL.get(
                    "val_cont_by_source", {}
                ).items()
            },
            "| sampled full causal loss:",
            None if LAST_VAL_DETAIL.get("val_full_causal") is None
            else round(LAST_VAL_DETAIL["val_full_causal"], 4),
        )

        append_log({
            "type": "val",
            "step": global_step,
            "tokens": int(cursor * SEQ_LEN),
            "val_cont": val_loss,
            "val_ppl": ppl,
            **LAST_VAL_DETAIL,
        })

        if val_loss < best_val:
            best_val = val_loss

            best_obj = {
                "model": model.state_dict(),
                "global_step": global_step,
                "val_loss": best_val,
                "config": {
                    "architecture": "efficient_gqa_rope_swiglu_decoder",
                    "vocab_size": len(tokenizer),
                    "d_model": D_MODEL,
                    "n_layers": N_LAYERS,
                    "n_heads": N_HEADS,
                    "n_kv_heads": N_KV_HEADS,
                    "rope_base": ROPE_BASE,
                    "ffn_dim": FFN_DIM,
                    "dropout": DROPOUT,
                    "max_positions": MAX_POSITIONS,
                    "seq_len": SEQ_LEN,
                    "prompt_p10_tokens": PROMPT_P10_TOKENS,
                    "prompt_p95_tokens": PROMPT_P95_TOKENS,
                    "expected_cont_tokens": EXPECTED_CONT_TOKENS,
                    "prompt_split_min": PROMPT_SPLIT_MIN,
                    "prompt_split_max": PROMPT_SPLIT_MAX,
                },
            }

            atomic_torch_save(best_obj, BEST_CKPT)
            atomic_copy(BEST_CKPT, OUTSIDE_BEST_CKPT)

            print("Saved new BEST checkpoint + refreshed:", OUTSIDE_BEST_CKPT)

    if (
        global_step % SAVE_EVERY == 0
        or cursor >= N_TRAIN_USED
        or (
            HF_CHECKPOINT_REPO
            and global_step % HF_UPLOAD_EVERY == 0
        )
    ):
        save_latest_and_expose()

    if (
        HF_CHECKPOINT_REPO
        and (
            global_step % HF_UPLOAD_EVERY == 0
            or cursor >= N_TRAIN_USED
        )
    ):
        maybe_upload_latest_to_hf()

atomic_torch_save(
    {
        "model": model.state_dict(),
        "global_step": global_step,
        "val_loss": best_val,
        "config": checkpoint_object()["config"],
    },
    FINAL_CKPT,
)

print("Training complete.")
print("Final checkpoint:", FINAL_CKPT)
print("Best validation loss:", best_val)
print("Latest downloadable copy:", OUTSIDE_LATEST_CKPT)
print("Best downloadable copy:", OUTSIDE_BEST_CKPT)
print("Total elapsed hours:", (time.time() - train_start) / 3600)


Fresh model built: 99.31M parameters
AdamW fused: True
Training on 260,247 of 260,247 blocks (~400M tokens) | run=eff_full
step=50/16,266 | blocks=800/260,247 | processed≈1.2M tok | loss=5.9563 | lr=4.62e-05 | 11,970 tok/s | ETA≈9.2h
step=100/16,266 | blocks=1,600/260,247 | processed≈2.5M tok | loss=4.6152 | lr=9.23e-05 | 12,310 tok/s | ETA≈9.0h
step=150/16,266 | blocks=2,400/260,247 | processed≈3.7M tok | loss=4.1790 | lr=1.38e-04 | 12,375 tok/s | ETA≈8.9h
step=200/16,266 | blocks=3,200/260,247 | processed≈4.9M tok | loss=4.0189 | lr=1.85e-04 | 12,417 tok/s | ETA≈8.8h
step=250/16,266 | blocks=4,000/260,247 | processed≈6.1M tok | loss=3.8489 | lr=2.31e-04 | 12,464 tok/s | ETA≈8.8h
step=300/16,266 | blocks=4,800/260,247 | processed≈7.4M tok | loss=3.6841 | lr=2.77e-04 | 12,301 tok/s | ETA≈8.9h
step=350/16,266 | blocks=5,600/260,247 | processed≈8.6M tok | loss=3.5325 | lr=3.00e-04 | 12,277 tok/s | ETA≈8.8h
step=400/16,266 | blocks=6,400/260,247 | processed≈9.8M tok | loss=3.4302 | lr=3.0

In [17]:
# Cell 12 — Reload BEST checkpoint + batched GQA KV-cache generation   (v2)
# v2 adds, on top of v1's generation:
#   * no-repeat n-gram blocking inside the generated text (stops greedy / low-temperature decoding from looping)
#   * several candidates per prompt from ONE prefill (cache is expanded after the prompt is processed)
#   * Minimum-Bayes-Risk (MBR) selection: among the candidates keep the one most similar (BERTScore) to the others
#   * a 128th word is only accepted once it is complete (v1 could cut it in the middle of a word)
best_obj = torch.load(
    BEST_CKPT if BEST_CKPT.exists() else FINAL_CKPT,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_obj["model"])
model.eval()

def mask_specials(logits, block_eos=True):
    logits = logits.clone()
    banned = [tokenizer.pad_token_id, tokenizer.unk_token_id, tokenizer.bos_token_id]
    if block_eos:
        banned.append(tokenizer.eos_token_id)
    logits[:, banned] = -float("inf")
    return logits

# Whitespace-only BPE pieces are structural: never repetition-penalise them.
WHITESPACE_TOKEN_IDS = []
for _tid in range(len(tokenizer)):
    try:
        _piece = tokenizer.decode([_tid])
        if _piece and _piece.strip() == "":
            WHITESPACE_TOKEN_IDS.append(_tid)
    except Exception:
        pass

REP_EXEMPT = torch.zeros(len(tokenizer), dtype=torch.bool, device=DEVICE)
if WHITESPACE_TOKEN_IDS:
    REP_EXEMPT[torch.tensor(WHITESPACE_TOKEN_IDS, device=DEVICE)] = True
print("whitespace-only token ids exempt from repetition penalty:", len(WHITESPACE_TOKEN_IDS))

def apply_rep_penalty(logits, generated, penalty):
    if penalty == 1.0 or generated is None or generated.size(1) == 0:
        return logits
    sc = logits.gather(1, generated)
    penalized = torch.where(sc < 0, sc * penalty, sc / penalty)
    adjusted = torch.where(REP_EXEMPT[generated], sc, penalized)
    return logits.scatter(1, generated, adjusted)


class NoRepeatNgram:
    """Bans any token that would complete an n-gram (of BPE tokens) already present in the generated text.
    Only the generated text is tracked: copying names / phrases from the prompt stays allowed."""
    def __init__(self, rows, n):
        assert n >= 2
        self.n = n
        self.hist = [[] for _ in range(rows)]
        self.seen = [dict() for _ in range(rows)]

    def update(self, new_tokens):
        k = self.n - 1
        for r, t in enumerate(new_tokens):
            h = self.hist[r]
            if len(h) >= k:
                self.seen[r].setdefault(tuple(h[len(h) - k:]), set()).add(t)
            h.append(t)

    def banned(self):
        k = self.n - 1
        rows, toks = [], []
        for r, h in enumerate(self.hist):
            if len(h) >= k:
                s = self.seen[r].get(tuple(h[len(h) - k:]))
                if s:
                    rows.extend([r] * len(s))
                    toks.extend(s)
        return rows, toks


def sample_next(logits, temperature=None, top_p=None, top_k=None):
    """temperature: float, or a (rows,) tensor where a value <= 0 means greedy for that row."""
    temperature = GEN_TEMPERATURE if temperature is None else temperature
    top_p = GEN_TOP_P if top_p is None else top_p
    top_k = GEN_TOP_K if top_k is None else top_k

    if not torch.is_tensor(temperature):
        if temperature <= 0:
            return logits.argmax(-1, keepdim=True)
        t = torch.full((logits.size(0),), float(temperature), device=logits.device)
    else:
        t = temperature.to(logits.device)

    greedy = t <= 0
    greedy_tok = logits.argmax(-1, keepdim=True)
    logits = logits / torch.where(greedy, torch.ones_like(t), t)[:, None]

    if top_k > 0:
        k = min(top_k, logits.size(-1))
        kth = torch.topk(logits, k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))

    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        pr = torch.softmax(sl, dim=-1)
        remove = (pr.cumsum(-1) - pr) > top_p
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf"))
        logits.scatter_(1, si, sl)

    sampled = torch.multinomial(torch.softmax(logits, -1), 1)
    return torch.where(greedy[:, None], greedy_tok, sampled)


GEN_COUNTERS = {"total": 0, "short": 0, "fallback": 0}

@torch.inference_mode()
def generate_candidates(
    prompts, n=1, include_greedy=False,
    target_words=TARGET_GENERATION_WORDS,
    temperature=None, top_p=None, rep_penalty=None, no_repeat=None,
):
    """Returns, for each prompt, a list of n candidate continuations (each exactly `target_words` words when
    the budget allows). The prompt is encoded once; with n>1 the KV cache is expanded n times afterwards.
    If include_greedy, candidate 0 of every prompt is decoded greedily."""
    temperature = GEN_TEMPERATURE if temperature is None else temperature
    top_p = GEN_TOP_P if top_p is None else top_p
    rep_penalty = GEN_REP_PENALTY if rep_penalty is None else rep_penalty
    no_repeat = GEN_NO_REPEAT_NGRAM if no_repeat is None else no_repeat

    B = len(prompts)
    R = B * n
    enc = [tokenizer.encode(p)[-PROMPT_KEEP_TOKENS:] for p in prompts]
    lens = [len(e) for e in enc]
    S_pad = max(lens)

    ids = torch.full((B, S_pad), tokenizer.pad_token_id, dtype=torch.long, device=DEVICE)
    valid = torch.zeros(B, S_pad, dtype=torch.bool, device=DEVICE)
    for b, e in enumerate(enc):                       # left padding: real tokens get positions 0..len-1
        ids[b, S_pad - len(e):] = torch.tensor(e, dtype=torch.long, device=DEVICE)
        valid[b, S_pad - len(e):] = True
    pos = (valid.long().cumsum(1) - 1).clamp(min=0)
    next_pos = valid.sum(1)

    hard_cap = max(1, min(GEN_MAX_NEW_TOKENS, MAX_POSITIONS - int(next_pos.max().item())))

    amp = torch.amp.autocast(
        device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")
    )
    with amp:
        h, cache = model.forward_hidden(ids, use_cache=True, position_ids=pos, key_valid=valid)
    last_h = h[:, -1, :]
    del h

    if n > 1:                                         # row b*n + j  <->  prompt b, candidate j
        cache = [(k.repeat_interleave(n, 0), v.repeat_interleave(n, 0)) for k, v in cache]
        valid = valid.repeat_interleave(n, 0)
        next_pos = next_pos.repeat_interleave(n, 0)
        last_h = last_h.repeat_interleave(n, 0)

    temp_rows = torch.full((R,), float(temperature), device=DEVICE)
    if include_greedy:
        temp_rows[0::n] = 0.0
    all_greedy = bool((temp_rows <= 0).all())

    nrep = NoRepeatNgram(R, no_repeat) if no_repeat and no_repeat >= 2 else None

    gen = torch.empty(R, 0, dtype=torch.long, device=DEVICE)
    done = [False] * R
    done_step = [None] * R
    texts = [""] * R

    for step in range(hard_cap):
        logits = model.project(last_h).float()
        logits = mask_specials(logits, block_eos=True)
        logits = apply_rep_penalty(logits, gen, rep_penalty)
        if nrep is not None:
            br, bt = nrep.banned()
            if br:
                logits[torch.tensor(br, device=DEVICE), torch.tensor(bt, device=DEVICE)] = -float("inf")

        if all_greedy:
            nt = logits.argmax(-1, keepdim=True)
        else:
            nt = sample_next(logits, temperature=temp_rows, top_p=top_p)

        gen = torch.cat([gen, nt], dim=1)
        valid = torch.cat([valid, torch.ones(R, 1, dtype=torch.bool, device=DEVICE)], dim=1)
        if nrep is not None:
            nrep.update(nt.squeeze(1).tolist())

        if (step + 1) % 8 == 0 or step + 1 == hard_cap:
            gen_cpu = gen.tolist()
            for r in range(R):
                if not done[r]:
                    text = tokenizer.decode(gen_cpu[r])
                    words = text.split()
                    # the last word counts only once it is complete (followed by whitespace) or a further word exists
                    if len(words) > target_words or (len(words) == target_words and text[-1:].isspace()):
                        done[r] = True
                        done_step[r] = step + 1
                        texts[r] = " ".join(words[:target_words])
            if all(done):
                break

        with amp:
            h, cache = model.forward_hidden(
                nt, past_key_values=cache, use_cache=True,
                position_ids=next_pos[:, None], key_valid=valid,
            )
        last_h = h[:, -1, :]
        next_pos = next_pos + 1

    gen_cpu = gen.tolist()
    for r in range(R):
        GEN_COUNTERS["total"] += 1
        if not done[r]:
            words = tokenizer.decode(gen_cpu[r]).split()
            texts[r] = " ".join(words[:target_words])
            GEN_COUNTERS["short"] += 1
        elif lens[r // n] + done_step[r] > SEQ_LEN:
            GEN_COUNTERS["fallback"] += 1

    return [texts[b * n:(b + 1) * n] for b in range(B)]


# ----------------------------------------------------------------------------- MBR selection
def _lcs_len(a, b):
    if not a or not b: return 0
    if len(a) < len(b): a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count()

def _rouge_l(p, r):
    pt, rt = p.split(), r.split()
    l = _lcs_len(pt, rt)
    if not l: return 0.0
    pr, rc = l / len(pt), l / len(rt)
    return 2 * pr * rc / (pr + rc)

# BERTScore evaluator helper — LOCAL VALIDATION ONLY.
# It is used by Cell 14 against held-out references, never to choose an official
# candidate at inference time.
_BERT_SCORER = None
def get_bert_scorer():
    global _BERT_SCORER
    if _BERT_SCORER is None:
        import bert_score
        _BERT_SCORER = bert_score.BERTScorer(
            lang="en", batch_size=64, device=str(DEVICE)
        )
    return _BERT_SCORER


def pairwise_similarity(cands_a, cands_b, utility="rougel"):
    """
    Candidate-consensus utility used INSIDE generation.
    Deliberately restricted to ROUGE-L/text overlap: no pretrained evaluator
    is allowed to influence official predictions.
    """
    if utility != "rougel":
        raise ValueError(
            "Generation-time MBR utility must be 'rougel'. "
            "BERTScore is evaluation-only in this notebook."
        )
    return [_rouge_l(a, b) for a, b in zip(cands_a, cands_b)]

def mbr_select(candidate_lists, utility="rougel"):
    """candidate_lists: list (per prompt) of candidate strings. Returns the index chosen for each prompt:
    the candidate with the highest mean ROUGE-L similarity to the other candidates."""
    A, Bc, where = [], [], []
    for p, cands in enumerate(candidate_lists):
        for i in range(len(cands)):
            for j in range(i + 1, len(cands)):
                A.append(cands[i]); Bc.append(cands[j]); where.append((p, i, j))
    sims = pairwise_similarity(A, Bc, utility) if A else []
    score = [[0.0] * len(c) for c in candidate_lists]
    for (p, i, j), s in zip(where, sims):
        score[p][i] += s
        score[p][j] += s
    return [max(range(len(c)), key=lambda i: score[p][i]) for p, c in enumerate(candidate_lists)]


def generate_texts(prompts, mbr_n=None, mbr_greedy=None, mbr_utility=None,
                   target_words=TARGET_GENERATION_WORDS, **kw):
    """One text per prompt. mbr_n>1 -> sample mbr_n candidates (optionally one greedy) and return the MBR choice.
    MBR only chooses among candidates that reached the full word budget (when at least one did)."""
    mbr_n = MBR_N if mbr_n is None else mbr_n
    mbr_greedy = MBR_INCLUDE_GREEDY if mbr_greedy is None else mbr_greedy
    mbr_utility = MBR_UTILITY if mbr_utility is None else mbr_utility
    cands = generate_candidates(
        prompts, n=max(1, mbr_n), include_greedy=(mbr_n > 1 and mbr_greedy),
        target_words=target_words, **kw,
    )
    if mbr_n <= 1:
        return [c[0] for c in cands]
    pools = [[t for t in c if len(t.split()) == target_words] or c for c in cands]
    pick = mbr_select(pools, mbr_utility)
    return [p[i] for p, i in zip(pools, pick)]

def generate_batch(prompts, target_words=TARGET_GENERATION_WORDS, **kw):      # v1-compatible name
    return generate_texts(prompts, target_words=target_words, **kw)

def generate_many(texts, desc="gen", verbose=True, **gen_kwargs):
    n = max(1, gen_kwargs.get("mbr_n", MBR_N))
    per_batch = max(1, min(GEN_BATCH_SIZE, MAX_DECODE_ROWS // n))
    order = sorted(range(len(texts)), key=lambda i: len(tokenizer.encode(texts[i])))
    out = [None] * len(texts)
    t0 = time.time()
    for s in range(0, len(order), per_batch):
        idx = order[s:s + per_batch]
        outs = generate_texts([texts[i] for i in idx], **gen_kwargs)
        for i, r in zip(idx, outs):
            out[i] = r
        if verbose:
            print(f"  {desc}: {min(s + per_batch, len(order))}/{len(order)} | {time.time()-t0:.0f}s", flush=True)
    return out


val_gen = json.load(open(VAL_GEN_PATH, encoding="utf-8"))
GEN_COUNTERS.update(total=0, short=0, fallback=0)

torch.manual_seed(GENERATION_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GENERATION_SEED)

t0 = time.time()
val_ids = list(val_gen.keys())
val_outs = generate_many([val_gen[k]["text"] for k in val_ids], desc="val")
local_predictions = {k: {"text": o} for k, o in zip(val_ids, val_outs)}

for k in val_ids[:3]:
    print(f"{k} | words={len(local_predictions[k]['text'].split())} | {local_predictions[k]['text'][:160]!r}")

print("generation counters:", GEN_COUNTERS)
json.dump(local_predictions, open(LOCAL_PRED, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Generation elapsed:", f"{time.time()-t0:.1f}s")
print("Saved:", LOCAL_PRED)


whitespace-only token ids exempt from repetition penalty: 12
  val: 16/153 | 8s
  val: 32/153 | 16s
  val: 48/153 | 23s
  val: 64/153 | 33s
  val: 80/153 | 42s
  val: 96/153 | 49s
  val: 112/153 | 58s
  val: 128/153 | 67s
  val: 144/153 | 75s
  val: 153/153 | 87s
sc_00 | words=128 | 'and Sons Ltd., was a joint venture partnership firm and that the parties agreed to sell 250 bales of the said property to the appellant and it was because of th'
sc_01 | words=128 | "court should not have interfered with the trial court's order and deciding the suit on the basis of evidence on record. 24. For the reasons aforesaid, the impug"
sc_02 | words=128 | 'and which otherwise are in the nature of civil liability, it may not be in the nature of criminal proceedings under the Code which is a very limited stage for a'
generation counters: {'total': 153, 'short': 0, 'fallback': 1}
Generation elapsed: 87.4s
Saved: /kaggle/working/cs781_effdecoder_100m/val_pred.json


In [18]:
# Cell 13 — Local Phase-1 evaluator: BERTScore + ROUGE-L + BLEU-4
def lcs_length(a,b):
    if not a or not b: return 0
    if len(a)<len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a):
        masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0)
        x=m|state
        y=(state<<1)|1
        state=x & ~(x-y)
    return state.bit_count()

def rouge_l_f1(pred,ref):
    pt,rt=pred.split(),ref.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    p,r=l/len(pt),l/len(rt)
    return 2*p*r/(p+r)

def ngrams(t,n):
    return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds,refs):
    clipped=[0]*4
    totals=[0]*4
    pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,5):
            pn,rn=ngrams(p,n),ngrams(r,n)
            totals[n-1]+=sum(pn.values())
            for g,c in pn.items():
                clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/4)

truth=json.load(open(VAL_REF_PATH,encoding="utf-8"))
pred=json.load(open(LOCAL_PRED,encoding="utf-8"))

if set(truth)!=set(pred):
    raise ValueError(f"ID mismatch: missing={set(truth)-set(pred)}, extra={set(pred)-set(truth)}")

ids=list(truth)
P=[pred[i]["text"].strip() for i in ids]
R=[truth[i]["text"].strip() for i in ids]

for i,p in zip(ids,P):
    if not p:
        raise ValueError(f"empty prediction for {i}")

rl=sum(rouge_l_f1(p,r) for p,r in zip(P,R))/len(P)
b4=bleu4([p.split() for p in P],[r.split() for r in R])
out={"ROUGE-L F1":rl,"BLEU-4":b4}

bert_per_sample=None
if not FAST_EVAL_NO_BERT:
    import bert_score
    _,_,f=bert_score.score(
        P,R,
        lang="en",
        batch_size=BERTSCORE_BATCH_SIZE,
        verbose=False,
        device=str(DEVICE),
    )
    bert_per_sample=f.tolist()
    out={"BERTScore F1":float(f.mean().item()),**out}

print("examples:",len(ids))
print("mean words/pred:",sum(len(p.split()) for p in P)/len(P))
print(json.dumps({k:round(v,6) for k,v in out.items()},indent=2))
# ---- per-source breakdown (ids look like "sc_00", "hc_03", ...) -> needed for the data-mixing analysis
def _src(i): return i.split("_")[0]
print("\nPer-source results:")
for s_ in sorted(set(_src(i) for i in ids)):
    ix = [k for k,i in enumerate(ids) if _src(i)==s_]
    row = {"n":len(ix), "ROUGE-L F1": round(sum(rouge_l_f1(P[k],R[k]) for k in ix)/len(ix), 4)}
    if bert_per_sample is not None:
        row["BERTScore F1"] = round(sum(bert_per_sample[k] for k in ix)/len(ix), 4)
    print(" ", s_, row)

# ---- append this decoding setting + scores to a log (use it for the decoding sweep table in the report)
with open(ROOT/"eval_log.jsonl","a") as _f:
    _f.write(json.dumps({"time":time.strftime("%Y-%m-%d %H:%M:%S"), "run":RUN_NAME,
                         "temperature":GEN_TEMPERATURE, "top_p":GEN_TOP_P, "top_k":GEN_TOP_K,
                         "rep_penalty":GEN_REP_PENALTY, "no_repeat":GEN_NO_REPEAT_NGRAM, "mbr_n":MBR_N, "n":len(ids),
                         **{k:float(v) for k,v in out.items()}}) + "\n")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


examples: 153
mean words/pred: 128.0
{
  "BERTScore F1": 0.837933,
  "ROUGE-L F1": 0.223601,
  "BLEU-4": 0.09767
}

Per-source results:
  acts {'n': 33, 'ROUGE-L F1': 0.2358, 'BERTScore F1': 0.8363}
  hc {'n': 40, 'ROUGE-L F1': 0.2273, 'BERTScore F1': 0.8356}
  sc {'n': 40, 'ROUGE-L F1': 0.2, 'BERTScore F1': 0.8332}
  tr {'n': 40, 'ROUGE-L F1': 0.2334, 'BERTScore F1': 0.8465}


In [19]:
# Cell 14 — Decoding sweep: every setting is scored with BERTScore (the Phase-1 primary metric)   (v2)
# Every candidate setting is scored against held-out references with BERTScore (evaluation only). ROUGE-L can rank settings differently from
# BERTScore, and the scorer is cheap for ~160 pairs, so every setting gets BERTScore here.
# The result is written to best_decode.json; Cell 15 picks it up automatically (USE_SWEEP_BEST in Cell 2).
# BERTScore is never used inside official decoding; MBR rows use ROUGE-L candidate consensus only.
#
# Reading the table: "dF1" is the paired difference to row 0 (your v1 setting) and "±" is 2 standard errors of that
# difference over the local prompts. A dF1 smaller than its ± is noise - do not read anything into it.

SWEEP = [
    dict(name="v1 default  T0.8",        temperature=0.8, top_p=0.95, rep_penalty=1.03),
    dict(name="greedy",                  temperature=0.0, top_p=1.00, rep_penalty=1.00),
    dict(name="greedy rep1.04",          temperature=0.0, top_p=1.00, rep_penalty=1.04),
    dict(name="greedy nr12",             temperature=0.0, top_p=1.00, rep_penalty=1.00, no_repeat=12),
    dict(name="T0.4 rep1.02 nr12",       temperature=0.4, top_p=0.95, rep_penalty=1.02, no_repeat=12),
    dict(name="T0.6 rep1.03 nr12",       temperature=0.6, top_p=0.95, rep_penalty=1.03, no_repeat=12),
    dict(name="MBR8 T0.8 +greedy nr12",  temperature=0.8, top_p=0.95, rep_penalty=1.00, no_repeat=12, mbr_n=8, mbr_greedy=True, mbr_utility="rougel"),
    dict(name="MBR8 T1.0 +greedy nr12",  temperature=1.0, top_p=0.95, rep_penalty=1.00, no_repeat=12, mbr_n=8, mbr_greedy=True, mbr_utility="rougel"),
]

truth_sw = json.load(open(VAL_REF_PATH, encoding="utf-8"))
val_gen_sw = json.load(open(VAL_GEN_PATH, encoding="utf-8"))
ids_sw = list(val_gen_sw)
prompts_sw = [val_gen_sw[i]["text"] for i in ids_sw]
R_sw = [truth_sw[i]["text"].strip() for i in ids_sw]

scorer = get_bert_scorer()
rows = []
for j, cfg in enumerate(SWEEP):
    kw = {k: v for k, v in cfg.items() if k != "name"}
    torch.manual_seed(SEED + 9100)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + 9100)

    t0 = time.time()
    outs = generate_many(prompts_sw, verbose=False, **kw)
    P_sw = [o.strip() for o in outs]
    per = scorer.score(P_sw, R_sw, verbose=False, batch_size=BERTSCORE_BATCH_SIZE)[2].tolist()

    row = {
        "idx": j, "name": cfg["name"], "cfg": kw,
        "BERTScore F1": float(np.mean(per)),
        "ROUGE-L": sum(rouge_l_f1(p, r) for p, r in zip(P_sw, R_sw)) / len(P_sw)
                   if "rouge_l_f1" in globals() else None,
        "n_not_128_words": sum(len(p.split()) != TARGET_GENERATION_WORDS for p in P_sw),
        "sec": time.time() - t0,
        "_per": per,
    }
    base = rows[0]["_per"] if rows else per
    d = np.array(per) - np.array(base)
    row["dF1"] = float(d.mean())
    row["dF1_2se"] = float(2 * d.std(ddof=1) / math.sqrt(len(d))) if len(d) > 1 and j > 0 else 0.0
    rows.append(row)
    print(f"[{j}] {row['name']:<26} BERTScore F1={row['BERTScore F1']:.4f}  dF1={row['dF1']:+.4f} ±{row['dF1_2se']:.4f}"
          f"  ROUGE-L={row['ROUGE-L'] if row['ROUGE-L'] is None else round(row['ROUGE-L'], 4)}"
          f"  not-128={row['n_not_128_words']}  {row['sec']:.0f}s", flush=True)

best_decode = max(rows, key=lambda r: r["BERTScore F1"])
print("\n✅ Best by BERTScore F1:", best_decode["name"], "->", round(best_decode["BERTScore F1"], 4),
      f"(dF1 vs v1 setting {best_decode['dF1']:+.4f} ±{best_decode['dF1_2se']:.4f})")

non_mbr = [r for r in rows if r["cfg"].get("mbr_n", 1) <= 1]
if best_decode["cfg"].get("mbr_n", 1) > 1 and non_mbr:
    b2 = max(non_mbr, key=lambda r: r["BERTScore F1"])
    print(f"   best single-sample setting: {b2['name']} ({b2['BERTScore F1']:.4f}); MBR adds "
          f"{best_decode['BERTScore F1'] - b2['BERTScore F1']:+.4f}. If that is within noise, the cheaper single-sample "
          f"setting is the safer choice: set MBR_N=1 and copy its values into Cell 2.")

json.dump({"name": best_decode["name"], "cfg": best_decode["cfg"], "BERTScore F1": best_decode["BERTScore F1"]},
          open(ROOT / "best_decode.json", "w"), indent=2)
json.dump([{k: v for k, v in r.items() if k != "_per"} for r in rows],
          open(ROOT / "sweep_results.json", "w"), indent=2)
with open(ROOT / "eval_log.jsonl", "a") as _f:
    for r in rows:
        _f.write(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "run": RUN_NAME, "kind": "sweep",
                             "n": len(P_sw), **{k: v for k, v in r.items() if k != "_per"}}) + "\n")
print("saved:", ROOT / "best_decode.json", "and", ROOT / "sweep_results.json")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[0] v1 default  T0.8           BERTScore F1=0.8383  dF1=+0.0000 ±0.0000  ROUGE-L=0.2213  not-128=1  117s
[1] greedy                     BERTScore F1=0.8134  dF1=-0.0249 ±0.0049  ROUGE-L=0.2192  not-128=1  122s
[2] greedy rep1.04             BERTScore F1=0.8196  dF1=-0.0187 ±0.0052  ROUGE-L=0.222  not-128=1  120s


KeyboardInterrupt: 

In [20]:
GEN_TEMPERATURE = 0.8
GEN_TOP_P = 0.95
GEN_REP_PENALTY = 1.03

RUN_OFFICIAL_TEST = True

print(GEN_TEMPERATURE, GEN_TOP_P, GEN_REP_PENALTY)
print("RUN_OFFICIAL_TEST =", RUN_OFFICIAL_TEST)

0.8 0.95 1.03
RUN_OFFICIAL_TEST = True


In [21]:
# Cell 15 — Official generation + automatic exact-128 repair + final ZIP validation   (v2)
# v2: uses the decoding setting chosen by Cell 14 (best_decode.json) when USE_SWEEP_BEST=True, otherwise the GEN_* /
#     MBR_* values from Cell 2; supports MBR; resumable partial file is keyed on the full decoding setting.
def resolve_decode_cfg():
    cfg = dict(temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P, rep_penalty=GEN_REP_PENALTY,
               no_repeat=GEN_NO_REPEAT_NGRAM, mbr_n=MBR_N, mbr_greedy=MBR_INCLUDE_GREEDY)
    p = ROOT / "best_decode.json"
    if USE_SWEEP_BEST and p.exists():
        saved = json.load(open(p))
        cfg.update(saved["cfg"])
        print(f"Decoding setting from sweep ({saved['name']}, local BERTScore {saved['BERTScore F1']:.4f}):")
    else:
        print("Decoding setting from Cell 2 (no sweep result used):")
    cfg.setdefault("mbr_n", 1)
    # Safety: official multi-candidate selection must not use a pretrained evaluator.
    cfg["mbr_utility"] = "rougel"
    print("  ", cfg)
    return cfg

if RUN_OFFICIAL_TEST:
    DECODE_CFG = resolve_decode_cfg()
    test_path = find_test_file()
    test = {str(k):v for k,v in json.load(open(test_path,encoding="utf-8")).items()}

    fingerprint = {
        "architecture":"efficient_gqa_rope_swiglu_decoder",
        "ckpt_step":best_obj.get("global_step"),
        "decode_cfg":DECODE_CFG,
        "top_k":GEN_TOP_K,
        "mbr_utility":MBR_UTILITY,
        "prompt_keep":PROMPT_KEEP_TOKENS,
        "max_positions":MAX_POSITIONS,
        "generation_seed":GENERATION_SEED,
    }
    partial_path = ROOT / "prediction_partial.json"

    def atomic_json(obj, path):
        tmp = Path(str(path) + ".tmp")
        with open(tmp,"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False)
        os.replace(tmp,path)

    predictions = {}
    if partial_path.exists():
        saved = json.load(open(partial_path,encoding="utf-8"))
        if saved.get("fingerprint") == fingerprint:
            predictions = saved.get("predictions",{})
            print(f"Resuming official generation: {len(predictions)} already done.")
        else:
            print("Ignoring stale partial file.")

    todo = [k for k in test if k not in predictions]
    todo.sort(key=lambda k:len(tokenizer.encode(test[k]["text"])))

    print("Official test file:", test_path)
    print(f"Official prompts: {len(test)} | to generate now: {len(todo)}")

    if not predictions:
        torch.manual_seed(GENERATION_SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(GENERATION_SEED)
        print("Official generation seed:", GENERATION_SEED)

    GEN_COUNTERS.update(total=0,short=0,fallback=0)
    t0=time.time()

    per_batch = max(1, min(GEN_BATCH_SIZE, MAX_DECODE_ROWS // max(1, DECODE_CFG["mbr_n"])))
    print("prompts per generation batch:", per_batch)

    for n,s in enumerate(range(0,len(todo),per_batch),1):
        chunk=todo[s:s+per_batch]
        outs=generate_texts([test[k]["text"] for k in chunk], **DECODE_CFG)

        for k,o in zip(chunk,outs):
            predictions[k]={"text":o}

        if n % OFFICIAL_PARTIAL_EVERY == 0 or s+per_batch >= len(todo):
            atomic_json(
                {"fingerprint":fingerprint,"predictions":predictions},
                partial_path,
            )

        print(
            f"{len(predictions)}/{len(test)} | elapsed={time.time()-t0:.0f}s",
            flush=True,
        )

    if set(predictions) != set(test):
        raise RuntimeError("Official test ID mismatch")

    bad_ids = [
        k for k,v in predictions.items()
        if len(v["text"].split()) != TARGET_GENERATION_WORDS
    ]

    if bad_ids:
        print("Repairing short IDs:", {
            k:len(predictions[k]["text"].split()) for k in bad_ids
        })

        for k in bad_ids:
            fixed=None
            for attempt in range(8):
                seed = GENERATION_SEED + 100000 + int(k)*17 + attempt
                torch.manual_seed(seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(seed)

                text = generate_texts([test[k]["text"]], **DECODE_CFG)[0]
                if len(text.split()) != TARGET_GENERATION_WORDS:   # last resort: plain sampling
                    text = generate_texts([test[k]["text"]], temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P,
                                          rep_penalty=GEN_REP_PENALTY, no_repeat=0, mbr_n=1)[0]
                if len(text.split()) == TARGET_GENERATION_WORDS:
                    fixed=text
                    print(f"  repaired {k} on attempt {attempt+1}")
                    break

            if fixed is None:
                raise RuntimeError(
                    f"Could not repair ID {k}; do NOT submit yet."
                )
            predictions[k]={"text":fixed}

    bad = {
        k:len(v["text"].split())
        for k,v in predictions.items()
        if len(v["text"].split()) != TARGET_GENERATION_WORDS
    }
    if bad:
        raise RuntimeError(f"Wrong word counts remain: {bad}")

    with open(OFFICIAL_PRED,"w",encoding="utf-8") as f:
        json.dump(predictions,f,ensure_ascii=False,indent=2)

    with zipfile.ZipFile(SUBMISSION_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
        z.write(OFFICIAL_PRED,arcname="prediction.json")

    with zipfile.ZipFile(SUBMISSION_ZIP,"r") as z:
        names=z.namelist()
        assert names == ["prediction.json"], names
        zipped=json.loads(z.read("prediction.json").decode("utf-8"))

    assert len(zipped)==len(test)==1000
    zipped_bad={
        k:len(v["text"].split())
        for k,v in zipped.items()
        if len(v["text"].split()) != TARGET_GENERATION_WORDS
    }
    assert not zipped_bad,zipped_bad

    print("generation counters:",GEN_COUNTERS)
    print("Predictions:",len(zipped))
    print("Min words:",min(len(v["text"].split()) for v in zipped.values()))
    print("Max words:",max(len(v["text"].split()) for v in zipped.values()))
    print("ZIP contents:",names)
    print("prediction.json:",OFFICIAL_PRED)
    print("submission.zip:",SUBMISSION_ZIP)
    print("✅ OFFICIAL SUBMISSION PASSED ALL STRUCTURAL CHECKS")
else:
    print("RUN_OFFICIAL_TEST=False — official generation intentionally skipped.")


Decoding setting from Cell 2 (no sweep result used):
   {'temperature': 0.8, 'top_p': 0.95, 'rep_penalty': 1.03, 'no_repeat': 0, 'mbr_n': 1, 'mbr_greedy': True, 'mbr_utility': 'rougel'}
Official test file: Phase_1_Pretraining/Test/test.json
Official test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json
Official prompts: 1000 | to generate now: 1000
Official generation seed: 5274
prompts per generation batch: 16
16/1000 | elapsed=7s
32/1000 | elapsed=15s
48/1000 | elapsed=22s
64/1000 | elapsed=30s
80/1000 | elapsed=38s
96/1000 | elapsed=45s
112/1000 | elapsed=52s
128/1000 | elapsed=59s
144/1000 | elapsed=66s
160/1000 | elapsed=73s
176/1000 | elapsed=81s
192/1000 | elapsed=88s
208/1000 | elapsed=96s
224/1000 | elapsed=103s
240/1000 | elapsed=111s
256/1000 | elapsed=118s
272/1000 | elapsed=126s
288/1000 | elapsed=135s
304/1000 | elapsed=142s
320/1000 | elapsed=149s
336/1000 |

In [22]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".pt") and ("eff_full_best" in file or "eff_full_latest" in file):
            print(os.path.join(root, file))

/kaggle/input/datasets/aaryan1234567890/phase1-model4-decoder-efficient/eff_full_best (6).pt
/kaggle/input/datasets/aaryan1234567890/phase1-model4-decoder-efficient/eff_full_latest (6).pt


In [23]:
import os
import torch

paths = [
    "/kaggle/input/datasets/aaryan1234567890/phase1-model4-decoder-efficient/eff_full_latest (6).pt",
    "/kaggle/input/datasets/aaryan1234567890/phase1-model4-decoder-efficient/eff_full_best (6).pt",
]

def inspect_checkpoint(path):
    print("\n" + "=" * 100)
    print("PATH:", path)

    assert os.path.exists(path), f"❌ File not found: {path}"

    obj = torch.load(path, map_location="cpu", weights_only=False)

    print("✅ LOADS CORRECTLY")
    print("Size GB:", round(os.path.getsize(path) / (1024**3), 3))
    print("Keys:", list(obj.keys()))

    print("global_step:", obj.get("global_step"))
    print("cursor:", obj.get("cursor"))
    print("best_val:", obj.get("best_val"))
    print("val_loss:", obj.get("val_loss"))

    if "config" in obj:
        print("\nCONFIG:")
        for k, v in obj["config"].items():
            print(f"  {k}: {v}")

    resume_keys = {
        "model",
        "optimizer",
        "scheduler",
        "scaler",
        "cursor",
        "global_step",
        "best_val",
        "python_random_state",
        "config",
    }

    missing = resume_keys - set(obj.keys())

    if not missing:
        print("\n✅ FULLY RESUMABLE TRAINING CHECKPOINT")
    else:
        print("\nℹ️ MODEL / BEST CHECKPOINT")
        print("Missing resume keys:", sorted(missing))

    # Basic weight sanity
    state = obj["model"]
    total_tensors = len(state)
    total_params = sum(x.numel() for x in state.values())

    bad = []
    for name, tensor in state.items():
        if torch.is_floating_point(tensor):
            if not torch.isfinite(tensor).all():
                bad.append(name)

    print("State tensors:", total_tensors)
    print("Stored model parameters:", f"{total_params/1e6:.2f}M")

    if bad:
        print("❌ NON-FINITE tensors:", bad[:10])
    else:
        print("✅ All floating-point model tensors are finite")

    return obj


for p in paths:
    inspect_checkpoint(p)


PATH: /kaggle/input/datasets/aaryan1234567890/phase1-model4-decoder-efficient/eff_full_latest (6).pt
✅ LOADS CORRECTLY
Size GB: 1.11
Keys: ['model', 'optimizer', 'scheduler', 'scaler', 'cursor', 'global_step', 'best_val', 'python_random_state', 'config']
global_step: 16266
cursor: 260247
best_val: 1.3664915735571412
val_loss: None

CONFIG:
  architecture: efficient_gqa_rope_swiglu_decoder
  vocab_size: 24576
  d_model: 640
  n_layers: 17
  n_heads: 10
  n_kv_heads: 2
  ffn_dim: 2048
  rope_base: 10000.0
  rms_eps: 1e-05
  dropout: 0.0
  max_positions: 2560
  seq_len: 1536
  prompt_p10_tokens: 1050
  prompt_p95_tokens: 1139
  expected_cont_tokens: 384
  prompt_split_min: 1050
  prompt_split_max: 1139
  loss_on_all_positions: True

✅ FULLY RESUMABLE TRAINING CHECKPOINT
State tensors: 155
Stored model parameters: 99.31M
✅ All floating-point model tensors are finite

PATH: /kaggle/input/datasets/aaryan1234567890/phase1-model4-decoder-efficient/eff_full_best (6).pt
✅ LOADS CORRECTLY
Size G

## Final checklist (v3 BEST)

Before pilot training the log should show:

```text
Vocab: 24576
SEQ_LEN: ~1536
Train token IDs: 400,000,000
Model parameters: ~99.31M
SELF-TEST PASSED
```

**Pilot:** keep `RUN_MODE = "pilot"` for ~100–150 steps and check loss, tokens/s and VRAM.
**Full run:** set `RUN_MODE = "full"` in Cell 10, rerun Cell 10 and Cell 11 (fresh weights are built automatically).

**After training:** Cell 12 → Cell 13 → Cell 14 (sweep, ~25–35 min on a T4) → Cell 15 with `RUN_OFFICIAL_TEST = True`.
Read the sweep table: a `dF1` smaller than its `±` is noise. If MBR is not clearly better than the best single-sample row, set `MBR_N = 1`
and `USE_SWEEP_BEST = False` in Cell 2 and copy that row's values instead.

Downloadable copies: `/kaggle/working/eff_full_latest.pt` (resumable) and `/kaggle/working/eff_full_best.pt` (validation-selected).
Keep this notebook, `tokenizer.json` and both `.pt` files together for later phases.


### Important decoding rule

Cell 14 may use BERTScore on the **held-out local references** to choose temperature / top-p /
repetition / no-repeat / optional ROUGE-consensus settings. This is metric evaluation only.

Cell 15 does **not** use RoBERTa/BERTScore to inspect or select official-test candidates.
If multi-candidate MBR wins locally, official MBR uses only ROUGE-L consensus among candidates.
